# 🎵 Minicurso MCP: Construindo um Servidor Musical com Model Context Protocol
**SECOMP UFPE — Semana de Computação**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Brunonaive03/mcp-course-secomp/blob/main/notebooks/minicurso_mcp.ipynb)

Seja muito bem-vindo ao laboratório prático de **Model Context Protocol (MCP)**!

Neste minicurso, você aprenderá na prática como transformar um modelo de linguagem (LLM) em um assistente autônomo conectado a dados e sistemas do mundo real. Construiremos juntos um **Servidor MCP de Catálogo Musical** integrado a uma base de dados MongoDB e uma interface de Chat interativa com **Player de Música estilo Spotify**.

---

### 🧠 O que é o Model Context Protocol (MCP)?
O **MCP** é um padrão aberto (open standard) criado pela Anthropic que padroniza a forma como modelos de inteligência artificial se comunicam com:
- **Fontes de dados** (bancos de dados, arquivos locais, repositórios GitHub, Google Drive).
- **Ferramentas externas** (APIs REST, consoles de execução, automações).

Antes do MCP, cada framework de IA (LangChain, LlamaIndex, agentes proprietários) inventava seu próprio formato para *tool calling* e carregamento de contexto. O MCP funciona como o **"USB-C para aplicações de IA"**: se você constrói um Servidor MCP, qualquer Host compatível (Claude Desktop, Cursor, Gradio, MCPAgent) consegue se conectar a ele imediatamente!

---

### 🏛️ A Arquitetura MCP em 3 Papéis:
1. **MCP Host**: A aplicação cliente onde o usuário interage (neste minicurso: nossa interface Gradio / agente Python).
2. **MCP Client**: O componente que gerencia a sessão de protocolo, descobre as ferramentas e envia as chamadas.
3. **MCP Server**: O serviço leve que você irá implementar, disponibilizando capacidades do catálogo musical.

---

### 🔱 A Trinca de Primitivas do MCP:
- 🛠️ **Tools (Ferramentas)**: Funções executáveis invocadas *pelo LLM*. O modelo decide autonomamente quando e como chamá-las para responder a uma pergunta.
- 📖 **Resources (Recursos)**: Dados ou mídias lidos *pela Aplicação/Host* através de URIs (ex: `music://tracks/{track_id}`). Permitem transferir dados detalhados (imagens, URLs de áudio) diretamente para a interface, sem poluir o raciocínio do modelo.
- 💡 **Prompts (Modelos)**: Modelos de mensagens parametrizáveis acionados *pelo Usuário* na interface para padronizar consultas frequentes.

---
## 📦 1. Instalação e Preparação do Ambiente

Vamos instalar o pacote `mcp-course-secomp` diretamente do GitHub. Ele contém o SDK do MCP, clientes de LLM, drivers do MongoDB e nossa interface gráfica em Gradio.

In [ ]:
# Instalar utilitários do minicurso diretamente do GitHub
%pip install git+https://github.com/Brunonaive03/mcp-course-secomp.git

### 🗄️ Inicializando o Banco de Dados MongoDB no Colab

Para simular um ambiente de produção real, o minicurso utiliza uma base MongoDB (`mcp_music_lab`) com dados reais de artistas como Queen, Oasis, Arctic Monkeys, etc., contendo:
- Metadados de faixas e álbuns.
- Amostras de **áudio de 30 segundos** (previews reais).
- Imagens de capa de alta resolução.

A função `prepare_environment()` automatiza todo o processo no Colab: instala os pacotes do MongoDB compatíveis com o Ubuntu da máquina, inicia o serviço `mongod` em background e restaura o catálogo de dados via `mongorestore`.

In [ ]:
from mcp_utils import prepare_environment

# Configura o MongoDB e popula o banco de dados 'mcp_music_lab'
prepare_environment()

---
## 🔑 2. Configurando a Chave da API da LLM

Neste minicurso, utilizaremos o modelo **`gpt-oss:20b`** (ou modelos compatíveis via Ollama Cloud / OpenAI API) para orquestrar as decisões do agente e o *tool calling*.

### Como configurar sua chave no Google Colab:
1. No menu lateral esquerdo do Colab, clique no ícone de chave (**Secrets** 🔑).
2. Adicione um novo segredo:
   - **Nome**: `OPENAI_API_KEY` (ou `OLLAMA_API_KEY`)
   - **Valor**: `sua-chave-aqui`
3. Ative a chave seletora **"Acesso ao notebook"** (Notebook access).

Vamos executar o teste abaixo para confirmar se a autenticação está funcionando:

In [ ]:
from mcp_utils.llm import get_openai_client, DEFAULT_MODEL

# Testa a inicialização do cliente e a leitura do segredo
try:
    client = get_openai_client()
    print("✅ Cliente LLM autenticado com sucesso!")
    print(f"🤖 Modelo padrão configurado: {DEFAULT_MODEL}")
except Exception as e:
    print(f"⚠️ Atenção ao configurar o segredo: {e}")

---
## 🛡️ 3. O Coração do FastMCP: Validação e Schemas com Pydantic

Antes de codificar as ferramentas, é fundamental entender **como o FastMCP transforma funções Python em capacidades compreensíveis por uma IA**.

No ecossistema Python do MCP (via classe `MCPServer`, anteriormente chamada `FastMCP`), todo o mapeamento de parâmetros é gerido pelo **Pydantic**.

```
    Função Python com Type Hints
                │
                ▼
    [Pydantic Type Inspector]
                │
                ▼
    JSON Schema padronizado (tools/list)
                │
                ▼
    Enviado ao LLM para Tool Calling
```

---

### ⚠️ Cuidados Fundamentais com o INPUT das Tools:

1. **Type Hints Obrigatórios**: O LLM não sabe o tipo dos parâmetros se você não declarar. Funções como `def search(artist, limit):` geram schemas vazios, levando o modelo a alucinar tipos (ex: passar uma string onde esperava-se um número). Declare sempre: `def search(artist: str, limit: int = 10):`.
2. **Docstrings Ricas**: O MCP extrai a descrição da tool e dos argumentos diretamente da *Docstring* da função (ou de `pydantic.Field(..., description="...")`). **A docstring é o "manual de instruções" que o modelo lê** para decidir quando deve invocar sua ferramenta.
3. **Validação Estrita**: Se o LLM tentar chamar sua função com argumentos inválidos, o Pydantic intercepta antes da execução e retorna um erro estruturado ao modelo, permitindo que ele se auto-corrija!

---

### ⚠️ Cuidados Fundamentais com o OUTPUT das Tools:

1. **Economia de Tokens**: A resposta de uma tool (`TextContent`) entra diretamente no **contexto de mensagens** da LLM. Se a sua ferramenta devolver um JSON de 50.000 tokens com base64 de áudios e imagens, o modelo pode estourar a janela de contexto ou ficar lento e caro!
2. **Regra de Ouro da Arquitetura MCP**:
   - **Tools** devem devolver apenas dados leves e essenciais para a LLM raciocinar (ex: ID da faixa, título, artista).
   - **Resources** devem ser usados para disponibilizar arquivos pesados, URLs de mídia e dados detalhados para a aplicação consumidora!

---
## 🛠️ 4. Construindo o Servidor MCP Musical

Agora vamos implementar o nosso servidor MCP completo. Ele conterá as três primitivas essenciais:
1. **Tool `search_tracks_by_artist`**: Busca faixas de um artista no catálogo MongoDB e retorna uma lista compacta.
2. **Tool `get_album_reviews`**: Consulta resenhas de críticos e fãs sobre álbuns no banco.
3. **Resource `music://tracks/{track_id}`**: Disponibiliza os metadados completos de uma faixa específica (incluindo URL de preview e capa).
4. **Prompt `discover_artist`**: Um modelo reutilizável de prompt para iniciar uma exploração musical estruturada.

In [ ]:
import json
from pymongo import MongoClient
from mcp.server.mcpserver import MCPServer
from pydantic import Field

# 1. Inicializar o Servidor MCP
server = MCPServer("mcp-music-lab")

# 2. Conectar à base de dados MongoDB local populada anteriormente
mongo = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=2000)["mcp_music_lab"]

# ==============================================================================
# PRIMITIVA 1: TOOLS (Ações solicitadas pelo LLM)
# ==============================================================================

@server.tool()
def search_tracks_by_artist(
    artist: str = Field(..., description="Nome do artista ou banda musical (ex: 'Oasis', 'Queen')"),
    limit: int = Field(5, ge=1, le=20, description="Quantidade máxima de faixas a retornar (entre 1 e 20)")
) -> str:
    """Busca faixas musicais de um artista específico no catálogo.
    
    Retorna uma lista resumida contendo o ID da faixa, título e artistas.
    """
    cursor = mongo["tracks"].find(
        {"artists": {"$regex": artist, "$options": "i"}},
        {"_id": 1, "title": 1, "artists": 1}
    ).limit(limit)

    results = [
        {"track_id": str(doc["_id"]), "title": doc.get("title", ""), "artists": doc.get("artists", [])}
        for doc in cursor
    ]

    if not results:
        return f"Nenhuma faixa encontrada para o artista '{artist}'."
    
    # Retorna JSON compacto para não poluir o contexto da LLM
    return json.dumps(results, ensure_ascii=False)


@server.tool()
def get_album_reviews(
    album_title: str = Field(..., description="Título aproximado do álbum para buscar resenhas")
) -> str:
    """Obtém avaliações e resenhas críticas sobre um determinado álbum."""
    cursor = mongo["reviews"].find(
        {"album_title": {"$regex": album_title, "$options": "i"}},
        {"_id": 0, "album_title": 1, "reviewer": 1, "score": 1, "comment": 1}
    ).limit(3)

    reviews = list(cursor)
    if not reviews:
        return f"Nenhuma resenha encontrada para o álbum '{album_title}'."
    
    return json.dumps(reviews, ensure_ascii=False)


# ==============================================================================
# PRIMITIVA 2: RESOURCES (Dados lidos pela Aplicação/Host via URI)
# ==============================================================================

@server.resource("music://tracks/{track_id}")
def get_track_resource(track_id: str) -> str:
    """Recurso acessado pelo Host para obter mídia (capa e áudio de preview 30s) da faixa."""
    track = mongo["tracks"].find_one({"_id": track_id})
    if track:
        return json.dumps(track, default=str)
    return "{}"


# ==============================================================================
# PRIMITIVA 3: PROMPTS (Modelos de instruções pré-definidos para o Usuário)
# ==============================================================================

@server.prompt()
def discover_artist(artist: str) -> str:
    """Gera um roteiro interativo para o usuário descobrir os sucessos de um artista."""
    return (
        f"Apresente uma breve introdução musical sobre o artista '{artist}'. "
        f"Em seguida, consulte o catálogo utilizando as ferramentas disponíveis, "
        f"liste as principais faixas encontradas e escolha a melhor para tocar no player!"
    )

print("🎵 Servidor MCP 'mcp-music-lab' registrado com sucesso!")

---
## 🔍 5. O Host em Ação: Descoberta de Capacidades

Vamos usar a classe `MCPAgent` do `mcp_utils` para simular como um cliente MCP se conecta ao servidor, inspeciona o catálogo de ferramentas e lê seus esquemas gerados automaticamente pelo Pydantic:

In [ ]:
from mcp_utils.agent import MCPAgent

# Cria uma instância de teste do agente conectado ao servidor em memória
agent = MCPAgent(server=server)

print("🛠️ Ferramentas Descobertas no Servidor:")
for tool in agent.get_available_tools():
    print(f" - Nome: {tool['name']}")
    print(f"   Descrição: {tool['description']}")
    print(f"   Schema Pydantic: {json.dumps(tool['parameters'], indent=2)}\n")

print("📖 Recursos Registrados:")
for res in agent.get_available_resources():
    print(f" - URI Template: {res['uri']}")

print("\n💡 Prompts Registrados:")
for prompt in agent.get_available_prompts():
    print(f" - Prompt: {prompt['name']} | Argumentos: {[a['name'] for a in prompt['arguments']]}")

---
## 🚀 6. Executando o MCP Music Chat com Player de Prévias

Chegou a hora de rodar a aplicação completa!

A função `launch_music_chat` iniciará uma interface Gradio com tema escuro estilo Spotify. Ela:
1. Conecta o `MCPAgent` ao seu `server`.
2. Habilita o LLM para invocar as ferramentas criadas.
3. Quando o modelo encontra músicas, o agente resolve os **MCP Resources** (`music://tracks/{track_id}`).
4. Carrega a capa do álbum, o áudio da prévia de 30 segundos e uma playlist interativa onde você pode alternar entre todas as faixas!
5. Disponibiliza um acordeão superior com os **Prompts MCP** registrados no servidor.

> **💡 Dica:** Ao executar a célula abaixo, será gerado um link público (`https://....gradio.live`). Clique nele para abrir o player em uma nova aba!

In [ ]:
from mcp_utils import launch_music_chat

# Inicia a interface Gradio interativa
launch_music_chat(server=server)

---
## 🎯 7. Desafios Práticos para o Laboratório

Agora é a sua vez de expandir o servidor MCP! Tente implementar os exercícios abaixo na célula seguinte:

### 🏆 Desafio 1: Criar uma Tool de Busca por Álbum com Pydantic
Crie a tool `search_tracks_by_album(album_title: str, limit: int = 5)`.
- Dica: consulte a coleção `mongo["tracks"]` filtrando pelo campo `album_title`.
- Use `Field(..., description="...")` para documentar bem os campos.

### 🏆 Desafio 2: Criar um Resource para Álbuns
Adicione um recurso `@server.resource("music://albums/{album_id}")` que busca os dados do álbum na coleção `mongo["albums"]`.

### 🏆 Desafio 3: Criar um Prompt de Batalha de Bandas
Crie um prompt `@server.prompt() def band_battle(band_a: str, band_b: str)` que instrua o LLM a comparar os estilos das duas bandas e recomendar faixas de ambas no catálogo!

In [ ]:
# Espaço para os alunos implementarem os desafios práticos:

# Exemplo de rascunho:
# @server.tool()
# def search_tracks_by_album(
#     album_title: str = Field(..., description="Nome do álbum"),
#     limit: int = Field(5, description="Limite de faixas")
# ) -> str:
#     # Sua implementação aqui...
#     pass